# Module 5: Multi-Mode Optics, Linear Networks & CV Quantum Information
## Notebook 5.4: Continuous-Variable Quantum Teleportation (Braunstein-Kimble)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_5_multimode_quantum_information/04_continuous_variable_quantum_teleportation.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** Light-Matter Interaction & Cavity QED
* **Module 4:** Open Quantum Systems, Dissipation & Trajectories
* **Module 5:** **Multi-Mode Optics, Linear Networks & CV Quantum Information**
  * **5.1** The Quantum Beam Splitter & Entanglement Generation
  * **5.2** Two-Photon Interference & The Hong-Ou-Mandel Effect
  * **5.3** Two-Mode Squeezing, EPR States & Continuous-Variable Inseparability
  * **5.4 Continuous-Variable Quantum Teleportation (Braunstein-Kimble)** $\leftarrow$ *(You are here)*
  * **5.5** Gaussian States, Symplectic Operations & Covariance Matrices
  * **5.6** Quantum State Engineering: Photon Addition, Subtraction & Catalysis
  * **5.7** Linear Optical Quantum Computing & Boson Sampling
* **Module 6:** Advanced Frontiers & Modern Computational Quantum Optics

---

### Learning Objectives
1. **The Braunstein-Kimble Protocol:** Understand the continuous-variable teleportation protocol using an Einstein-Podolsky-Rosen (EPR) entangled state, joint homodyne Bell measurement, and feedforward phase-space displacement.
2. **Phase-Space Noise Kernel:** Derive the teleportation output relation $\hat{x}_{\text{out}} = \hat{x}_{\text{in}} + \delta x(r)$ and prove that the excess quantum noise variance scales as $\sigma_{\text{noise}}^2 = e^{-2r}$.
3. **The Classical Teleportation Bound:** Demonstrate why any classical measure-and-prepare strategy is fundamentally capped at fidelity $\mathcal{F}_{\text{class}} \le 1/2$ for coherent states.
4. **Quantum Advantage & High-Fidelity Recovery:** Verify that shared two-mode squeezing ($r > 0$) surpasses the classical bound: $\mathcal{F}(r) = \frac{1}{1 + e^{-2r}} > 1/2$.
5. **Non-Gaussian State Teleportation:** Analyze the survival of quantum non-classicality (such as Wigner negativity $W(0, 0) < 0$) when teleporting single-photon and Schrödinger cat states across noisy quantum channels.



In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
from scipy.signal import convolve2d
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "lines.linewidth": 1.8,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

# Verify environment capabilities
print(f"NumPy version: {np.__version__}")
try:
    import qutip as qt
    print(f"QuTiP version: {qt.__version__} (Available for optional comparison)")
    HAS_QUTIP = True
except ImportError:
    print("QuTiP not installed; running pure, optimized native NumPy/SciPy engine.")
    HAS_QUTIP = False



---

## Theoretical Framework: The Braunstein-Kimble Teleportation Protocol

### 1. Quantum Teleportation in Continuous Variables
In 1993, Bennett et al. conceived quantum teleportation for discrete qubits using Bell pairs. 
In 1998, Samuel L. Braunstein and H. Jeff Kimble generalized quantum teleportation to **infinite-dimensional continuous-variable (CV) systems**, experimentally realized that same year by Furusawa, Sorensen, Braunstein, Fuchs, Kimble, and Polzik.

Instead of teleporting a discrete spin, the goal is to transfer an unknown continuous quantum optical state $|\psi_{\text{in}}\rangle$ (such as an arbitrary coherent, squeezed, or non-Gaussian state) from Alice to Bob using:
1. A pre-shared **Two-Mode Squeezed Vacuum (TMSV)** state with squeezing parameter $r$.
2. Linear beam splitters and **balanced homodyne detectors**.
3. A classical communication channel.
4. An electro-optic feedforward **phase-space displacement**.

---

### 2. Step-by-Step Mathematical Walkthrough

#### Step A: The Shared EPR Entangled Resource
Alice and Bob share an EPR entangled state $|\text{TMSV}(r)\rangle$ with modes $A$ (held by Alice) and $B$ (held by Bob):
$$\hat{x}_A = \hat{x}_{0, A}\cosh r + \hat{x}_{0, B}\sinh r, \quad \hat{x}_B = \hat{x}_{0, B}\cosh r + \hat{x}_{0, A}\sinh r$$
$$\hat{p}_A = \hat{p}_{0, A}\cosh r - \hat{p}_{0, B}\sinh r, \quad \hat{p}_B = \hat{p}_{0, B}\cosh r - \hat{p}_{0, A}\sinh r$$

#### Step B: Alice's Continuous Bell Measurement
Alice interferes the unknown input state $\hat{a}_{\text{in}}$ and her half of the EPR pair $\hat{a}_A$ on a balanced 50:50 beam splitter:
$$\hat{u} = \hat{x}_{\text{in}} - \hat{x}_A, \quad \hat{v} = \hat{p}_{\text{in}} + \hat{p}_A$$

Alice simultaneously measures $\hat{u}$ and $\hat{v}$ using two balanced homodyne detectors. 
Because $[\hat{u}, \hat{v}] = [\hat{x}_{\text{in}}, \hat{p}_{\text{in}}] - [\hat{x}_A, \hat{p}_A] = i - i \equiv 0$, these two operators commute and can be measured **simultaneously without quantum backaction**! 
The measurement yields two classical real numbers $(u, v) \in \mathbb{R}^2$.

#### Step C: Classical Communication
Alice transmits the pair $(u, v)$ to Bob over a classical channel (e.g., optical fiber or radio).

#### Step D: Bob's Feedforward Displacement
Bob receives $(u, v)$ and applies a unitary phase-space displacement $\hat{D}_B(\beta)$ to his mode $B$, with $\beta = \frac{u + iv}{\sqrt{2}}$:
$$\hat{x}_B \to \hat{x}_{\text{out}} = \hat{x}_B + u = \hat{x}_B + (\hat{x}_{\text{in}} - \hat{x}_A)$$
$$\hat{p}_B \to \hat{p}_{\text{out}} = \hat{p}_B + v = \hat{p}_B + (\hat{p}_{\text{in}} + \hat{p}_A)$$

Regrouping terms:
$$\hat{x}_{\text{out}} = \hat{x}_{\text{in}} + (\hat{x}_B - \hat{x}_A) = \hat{x}_{\text{in}} + \sqrt{2}\hat{x}_-$$
$$\hat{p}_{\text{out}} = \hat{p}_{\text{in}} + (\hat{p}_B + \hat{p}_A) = \hat{p}_{\text{in}} + \sqrt{2}\hat{p}_+$$

---

### 3. Noise Analysis & Squeezing Scaling
From Notebook 5.3, the relative EPR variances in the TMSV state are:
$$\langle (\sqrt{2}\hat{x}_-)^2 \rangle = e^{-2r}, \quad \langle (\sqrt{2}\hat{p}_+)^2 \rangle = e^{-2r}$$

Therefore, the output quadratures at Bob's station are:
$$\hat{x}_{\text{out}} = \hat{x}_{\text{in}} + \delta x(r), \quad \hat{p}_{\text{out}} = \hat{p}_{\text{in}} + \delta p(r)$$
where $\delta x(r)$ and $\delta p(r)$ are independent Gaussian quantum noise fluctuations with variance:
$$\sigma_{\text{noise}}^2 = e^{-2r}$$

---

### 4. Teleportation Fidelity & The Classical Bound
For an arbitrary coherent state input $|\alpha_{\text{in}}\rangle$, the teleportation fidelity is given by the overlap:
$$\mathcal{F} = \langle\alpha_{\text{in}}|\hat{\rho}_{\text{out}}|\alpha_{\text{in}}\rangle = \frac{1}{1 + \sigma_{\text{noise}}^2} = \frac{1}{1 + e^{-2r}}$$

#### Fundamental Thresholds:
1. **The Classical Limit ($r = 0$):**
   If Alice and Bob share no entanglement ($r = 0$, vacuum state), the homodyne measurement noise adds exactly one quantum of vacuum noise ($\sigma_{\text{noise}}^2 = 1.0$):
   $$\mathcal{F}_{\text{classical}} = \frac{1}{1 + 1} = \frac{1}{2} = 50\%$$
   Any protocol operating without entanglement cannot exceed $\mathcal{F} \le 1/2$.
2. **Quantum Teleportation ($r > 0$):**
   As soon as squeezing $r > 0$ is shared, $\mathcal{F} > 1/2$.
   For $r = 1.0$ ($8.69\text{ dB}$ of squeezing): $\mathcal{F} = \frac{1}{1 + e^{-2}} \approx 88.1\%$.
3. **Perfect Teleportation ($r \to \infty$):**
   $$\lim_{r \to \infty} \sigma_{\text{noise}}^2 = 0 \implies \lim_{r \to \infty} \mathcal{F} = 1.0$$



In [ ]:
# Implementation: High-Precision Braunstein-Kimble CV Teleportation Engine

class CVTeleportationEngine:
    """
    Simulation engine for Continuous-Variable Quantum Teleportation (Braunstein-Kimble protocol).
    Simulates input states, EPR channel noise, homodyne Bell measurements, and fidelity calculations.
    """
    def __init__(self, squeezing_r=1.0):
        self.r = float(squeezing_r)
        self.noise_variance = np.exp(-2.0 * self.r)

    def teleport_quadratures(self, x_in, p_in, n_shots=10000, seed=42):
        """
        Simulate stochastic homodyne measurement and feedforward for input quadratures.
        x_out = x_in + delta_x, where delta_x ~ N(0, exp(-2r))
        p_out = p_in + delta_p, where delta_p ~ N(0, exp(-2r))
        """
        rng = np.random.default_rng(seed)
        std_noise = np.sqrt(self.noise_variance)
        
        delta_x = rng.normal(0.0, std_noise, size=n_shots)
        delta_p = rng.normal(0.0, std_noise, size=n_shots)
        
        x_out = x_in + delta_x
        p_out = p_in + delta_p
        
        return {
            "x_out": x_out,
            "p_out": p_out,
            "delta_x": delta_x,
            "delta_p": delta_p,
            "mean_x": float(np.mean(x_out)),
            "mean_p": float(np.mean(p_out)),
            "var_noise_x": float(np.var(delta_x)),
            "var_noise_p": float(np.var(delta_p))
        }

    def coherent_state_fidelity(self, squeezing_r=None):
        """
        Theoretical fidelity for teleporting an arbitrary coherent state:
            F(r) = 1 / (1 + exp(-2*r))
        """
        if squeezing_r is None:
            r_val = self.r
        elif np.isscalar(squeezing_r):
            r_val = float(squeezing_r)
        else:
            r_val = np.asarray(squeezing_r, dtype=float)
            
        return 1.0 / (1.0 + np.exp(-2.0 * r_val))

    def wigner_convolution(self, W_in, x_grid, p_grid, squeezing_r=None):
        """
        Compute output Wigner function via 2D Gaussian convolution with the channel noise kernel.
        """
        r_val = self.r if squeezing_r is None else float(squeezing_r)
        var = np.exp(-2.0 * r_val)
        
        dx = x_grid[1] - x_grid[0]
        dp = p_grid[1] - p_grid[0]
        
        # Gaussian noise kernel
        X, P = np.meshgrid(x_grid, p_grid)
        kernel = (1.0 / (np.pi * var)) * np.exp(-(X**2 + P**2) / var)
        kernel /= np.sum(kernel * dx * dp)
        
        # 2D Convolution using SciPy
        W_out = convolve2d(W_in, kernel, mode='same') * dx * dp
        return W_out

print("CVTeleportationEngine successfully loaded.")



In [ ]:
# Publication-Quality 4-Panel Visualization: CV Teleportation Fidelity & Phase-Space Recovery

engine = CVTeleportationEngine(squeezing_r=1.0)

# 1. Teleportation fidelity vs. squeezing parameter r and squeezing in dB
r_scan = np.linspace(0.0, 2.5, 100)
dB_scan = 8.686 * r_scan
fidelity_curve = engine.coherent_state_fidelity(r_scan)

# 2. Quadrature scatter for coherent state |alpha = 2.0 + 1.5i>
alpha_target = 2.0 + 1.5j
# quadratures: x = sqrt(2)*Re(alpha), p = sqrt(2)*Im(alpha)
x_target = np.sqrt(2.0) * np.real(alpha_target)  # ~ 2.828
p_target = np.sqrt(2.0) * np.imag(alpha_target)  # ~ 2.121

# Classical teleportation (r = 0, vacuum noise var = 1.0)
engine_class = CVTeleportationEngine(squeezing_r=0.0)
sim_class = engine_class.teleport_quadratures(x_target, p_target, n_shots=2000, seed=101)

# Quantum teleportation (r = 1.2, noise var = exp(-2.4) ~ 0.09)
engine_quantum = CVTeleportationEngine(squeezing_r=1.2)
sim_quantum = engine_quantum.teleport_quadratures(x_target, p_target, n_shots=2000, seed=202)

# 3. Wigner function cross-section for single photon Fock state |1>
# W_in(x, 0) = (1/pi) * (2*x^2 - 1) * exp(-x^2)
x_axis = np.linspace(-3.5, 3.5, 201)
p_axis = np.linspace(-3.5, 3.5, 201)
X, P = np.meshgrid(x_axis, p_axis)
W_single_photon = (1.0 / np.pi) * (2.0 * (X**2 + P**2) - 1.0) * np.exp(-(X**2 + P**2))

# Convolve with r = 0 (classical), r = 0.5, r = 1.2
W_out_r0 = engine.wigner_convolution(W_single_photon, x_axis, p_axis, squeezing_r=0.0)
W_out_r05 = engine.wigner_convolution(W_single_photon, x_axis, p_axis, squeezing_r=0.5)
W_out_r12 = engine.wigner_convolution(W_single_photon, x_axis, p_axis, squeezing_r=1.2)

# Generate 4-Panel Masterpiece Figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -------------------------------------------------------------
# Panel (a): Teleportation Fidelity vs. Squeezing
# -------------------------------------------------------------
ax_a = axes[0, 0]
ax_a.plot(r_scan, fidelity_curve, color="#2980b9", linewidth=2.5, label=r"Quantum Fidelity $\mathcal{F}(r) = \frac{1}{1 + e^{-2r}}$")
ax_a.axhline(0.5, color="#e74c3c", linestyle="--", linewidth=2.0, label=r"Classical Bound $\mathcal{F}_{\rm class} = 0.5$")

# Shaded quantum advantage region
ax_a.fill_between(r_scan, 0.5, fidelity_curve, color="#2ecc71", alpha=0.25, label=r"Quantum Advantage Region ($\mathcal{F} > 0.5$)")
ax_a.scatter([1.0], [engine.coherent_state_fidelity(1.0)], color="#8e44ad", s=80, zorder=5)
ax_a.annotate(f"r = 1.0 (8.7 dB)\nF = {engine.coherent_state_fidelity(1.0)*100:.1f}%", 
              xy=(1.0, engine.coherent_state_fidelity(1.0)), xytext=(1.2, 0.72),
              arrowprops=dict(arrowstyle="->", color="#8e44ad", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#f4ecf7", ec="#8e44ad", lw=1.2),
              fontweight="bold")

ax_a.set_title(r"(a) Coherent State Teleportation Fidelity", fontweight="bold")
ax_a.set_xlabel(r"Squeezing Parameter $r$")
ax_a.set_ylabel(r"Teleportation Fidelity $\mathcal{F}$")
ax_a.set_ylim(0.45, 1.02)
ax_a.legend(loc="lower right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (b): Phase Space Quadrature Recovery
# -------------------------------------------------------------
ax_b = axes[0, 1]
ax_b.scatter(sim_class["x_out"][:800], sim_class["p_out"][:800], color="#bdc3c7", alpha=0.4, s=15, label=r"Classical Teleportation ($r=0$)")
ax_b.scatter(sim_quantum["x_out"][:800], sim_quantum["p_out"][:800], color="#3498db", alpha=0.6, s=15, label=r"Quantum Teleportation ($r=1.2$)")
ax_b.scatter([x_target], [p_target], color="#e74c3c", marker="*", s=160, zorder=6, label=r"Exact Target $(x_{\rm in}, p_{\rm in})$")

# Draw 1-sigma uncertainty circles
circle_class = plt.Circle((x_target, p_target), 1.0, color="#7f8c8d", fill=False, linestyle="--", linewidth=1.8)
circle_quant = plt.Circle((x_target, p_target), np.sqrt(engine_quantum.noise_variance), color="#2980b9", fill=False, linewidth=2.0)
ax_b.add_patch(circle_class)
ax_b.add_patch(circle_quant)

ax_b.set_title(r"(b) Phase-Space Quadrature Recovery Scatter", fontweight="bold")
ax_b.set_xlabel(r"Position Quadrature $\hat{x}$")
ax_b.set_ylabel(r"Momentum Quadrature $\hat{p}$")
ax_b.set_xlim(0.0, 5.5)
ax_b.set_ylim(-0.5, 5.0)
ax_b.set_aspect("equal")
ax_b.legend(loc="upper left", framealpha=0.9)

# -------------------------------------------------------------
# Panel (c): Single-Photon Wigner Function Cross-Section
# -------------------------------------------------------------
ax_c = axes[1, 0]
mid_idx = len(p_axis) // 2

ax_c.plot(x_axis, W_single_photon[mid_idx, :], color="#2c3e50", linewidth=2.4, label=r"Input $|1\rangle$ (Ideal, $W(0)<0$)")
ax_c.plot(x_axis, W_out_r12[mid_idx, :], color="#27ae60", linewidth=2.0, label=r"Teleported ($r = 1.2$, Negative)")
ax_c.plot(x_axis, W_out_r05[mid_idx, :], color="#f39c12", linewidth=2.0, label=r"Teleported ($r = 0.5$)")
ax_c.plot(x_axis, W_out_r0[mid_idx, :], color="#e74c3c", linestyle="--", linewidth=1.8, label=r"Classical ($r = 0$, Positive)")

ax_c.axhline(0.0, color="gray", linestyle=":", linewidth=1.2)
ax_c.annotate("Wigner Negativity Preserved\nfor $r > 0.35$", xy=(0.0, W_out_r12[mid_idx, mid_idx]), xytext=(-2.8, -0.2),
              arrowprops=dict(arrowstyle="->", color="#27ae60", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#e8f8f5", ec="#27ae60", lw=1.2),
              fontweight="bold")

ax_c.set_title(r"(c) Wigner Cross-Section $W(x, 0)$ for Single Photon $|1\rangle$", fontweight="bold")
ax_c.set_xlabel(r"Position Quadrature $x$")
ax_c.set_ylabel(r"Wigner Function $W(x, 0)$")
ax_c.legend(loc="upper right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (d): Excess Noise Variance vs. Squeezing (dB)
# -------------------------------------------------------------
ax_d = axes[1, 1]
ax_d.semilogy(dB_scan, np.exp(-2.0 * r_scan), color="#8e44ad", linewidth=2.4, label=r"Channel Noise $\sigma_{\rm noise}^2 = e^{-2r}$")
ax_d.axhline(1.0, color="#e74c3c", linestyle="--", linewidth=1.8, label=r"Classical Vacuum Limit $\sigma^2 = 1.0$")

ax_d.set_title(r"(d) Exponential Noise Suppression vs. Squeezing", fontweight="bold")
ax_d.set_xlabel(r"Channel Squeezing (dB)")
ax_d.set_ylabel(r"Excess Noise Variance $\sigma_{\rm noise}^2$")
ax_d.set_ylim(0.003, 1.5)
ax_d.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()



> **Physical Insight: The No-Cloning Theorem and the Classical Fidelity Limit $\mathcal{F} = 1/2$**
>
> 1. **Why Can't Classical Teleportation Exceed 50% Fidelity?** Suppose Alice tries to teleport a coherent state $|\alpha\rangle$ without entanglement. She must measure $\hat{x}$ and $\hat{p}$ simultaneously. By the Heisenberg uncertainty principle $[\hat{x}, \hat{p}] = i$, an optimal simultaneous measurement (heterodyne detection) inevitably introduces an extra quantum of vacuum noise into the measurement record. When Bob reconstructs the state using this noisy classical information, the overlap between the target and reconstructed state cannot exceed:
>    $$\mathcal{F}_{\text{classical}} \le \frac{1}{2}$$
>    This classical cap is an exact consequence of the **quantum no-cloning theorem**: if a classical measurement could achieve $\mathcal{F} > 1/2$, Alice could broadcast the measurement record to multiple recipients, creating multiple high-fidelity clones of an unknown quantum state!
> 2. **How Entanglement Circumvents Heisenberg:** By utilizing a pre-shared Two-Mode Squeezed Vacuum, Alice does not measure $\hat{x}_{\text{in}}$ directly. Instead, she measures the **joint relative quadrature** $\hat{u} = \hat{x}_{\text{in}} - \hat{x}_A$ and $\hat{v} = \hat{p}_{\text{in}} + \hat{p}_A$. Because these commute ($[\hat{u}, \hat{v}] = 0$), Alice's Bell measurement is **completely noiseless**! The only noise in the protocol comes from the finite squeezing of the EPR pair ($\sigma_{\text{noise}}^2 = e^{-2r}$), allowing quantum teleportation to beat the classical $50\%$ bound.
> 3. **Preserving Quantum Negativity:** For non-Gaussian states (like single photons $|1\rangle$ or Schrödinger cats), the negative values of the Wigner function are fragile. Because the teleportation channel acts as a Gaussian convolution filter with variance $e^{-2r}$, Wigner negativity survives if and only if the squeezing exceeds a critical threshold: $r > \frac{1}{2}\ln 2 \approx 0.347\text{ nats}$ ($3.01\text{ dB}$).



In [ ]:
# Interactive Exploration: Braunstein-Kimble CV Teleportation Simulator

def interactive_teleportation_explorer(squeezing_dB=8.0, alpha_real=2.0, alpha_imag=1.0):
    """
    Interactive exploration of CV teleportation fidelity and noise suppression.
    """
    r_val = squeezing_dB / 8.686
    engine_sim = CVTeleportationEngine(squeezing_r=r_val)
    
    alpha_target = complex(alpha_real, alpha_imag)
    x_t = np.sqrt(2.0) * alpha_real
    p_t = np.sqrt(2.0) * alpha_imag
    
    res = engine_sim.teleport_quadratures(x_t, p_t, n_shots=1500, seed=42)
    fidelity = engine_sim.coherent_state_fidelity(r_val)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Subplot 1: Phase space scatter
    ax1.scatter(res["x_out"], res["p_out"], color="#3498db", alpha=0.5, s=15, label="Teleported Shots")
    ax1.scatter([x_t], [p_t], color="#e74c3c", marker="*", s=150, zorder=5, label="Target")
    
    # Uncertainty circle
    circ = plt.Circle((x_t, p_t), np.sqrt(engine_sim.noise_variance), color="#2c3e50", fill=False, linewidth=1.8)
    ax1.add_patch(circ)
    
    ax1.set_title(f"Phase Space Teleportation ($r = {r_val:.2f}$, {squeezing_dB:.1f} dB)\nNoise Var = {engine_sim.noise_variance:.3f}")
    ax1.set_xlabel(r"Quadrature $\hat{x}$")
    ax1.set_ylabel(r"Quadrature $\hat{p}$")
    ax1.set_xlim(x_t - 3.0, x_t + 3.0)
    ax1.set_ylim(p_t - 3.0, p_t + 3.0)
    ax1.set_aspect("equal")
    ax1.legend(loc="upper left")
    
    # Subplot 2: Fidelity gauge
    r_axis = np.linspace(0.0, 2.0, 100)
    ax2.plot(r_axis * 8.686, engine_sim.coherent_state_fidelity(r_axis), color="#2980b9", linewidth=2.2)
    ax2.axhline(0.5, color="#e74c3c", linestyle="--", label="Classical Cap (0.50)")
    ax2.scatter([squeezing_dB], [fidelity], color="#27ae60", s=100, zorder=5, label=f"Current: F = {fidelity*100:.1f}%")
    
    ax2.set_title(f"Coherent State Fidelity F = {fidelity*100:.1f}%\nQuantum Advantage: {fidelity > 0.5}")
    ax2.set_xlabel("Squeezing Level (dB)")
    ax2.set_ylabel("Fidelity F")
    ax2.set_ylim(0.45, 1.0)
    ax2.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

# Test invocation
interactive_teleportation_explorer(squeezing_dB=8.0, alpha_real=2.0, alpha_imag=1.0)

try:
    import ipywidgets as widgets
    from IPython.display import display
    w = widgets.interactive(
        interactive_teleportation_explorer,
        squeezing_dB=widgets.FloatSlider(value=8.0, min=0.0, max=16.0, step=0.5, description="Squeeze dB:"),
        alpha_real=widgets.FloatSlider(value=2.0, min=-3.0, max=3.0, step=0.5, description=r"Re($\alpha$):"),
        alpha_imag=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.5, description=r"Im($\alpha$):")
    )
    display(w)
except Exception:
    pass



---

## Guided Computational Challenges & Exercises

### Challenge 1: The Classical Teleportation Bound Benchmark
1. Prove analytically that without entanglement ($r = 0$), the Braunstein-Kimble teleportation protocol yields an added noise variance of $\sigma_{\text{noise}}^2 = 1.0$.
2. For an arbitrary input coherent state, assert that the classical fidelity is strictly:
   $$\mathcal{F}_{\text{classical}} \equiv \frac{1}{2} = 0.500000$$
3. Verify that `coherent_state_fidelity(0.0) == 0.5` to within $10^{-15}$.

---

### Challenge 2: Quantum Fidelity Scaling Benchmark
1. For squeezing $r = 1.0$ ($8.686\text{ dB}$), evaluate the exact theoretical fidelity:
   $$\mathcal{F}(1.0) = \frac{1}{1 + e^{-2}} \approx 0.880797$$
2. Simulate $50,000$ teleportation shots and assert that the empirical noise variance matches $e^{-2}$ to within $2\%$ relative error.
3. Assert that the analytical fidelity matches $0.880797$ to within machine precision.

---

### Challenge 3: Wigner Negativity Survival Threshold
1. For an input single-photon Fock state $|1\rangle$, the central Wigner value is $W(0, 0) = -1/\pi$.
2. Show that under Gaussian convolution with noise variance $\sigma^2 = e^{-2r}$, the central Wigner value transforms as:
   $$W_{\text{out}}(0, 0) = \frac{1}{\pi} \frac{\sigma^2 - 1}{(\sigma^2 + 1)^2} = \frac{1}{\pi} \frac{e^{-2r} - 1}{(e^{-2r} + 1)^2}$$
3. Prove that Wigner negativity $W_{\text{out}}(0, 0) < 0$ survives if and only if $r > 0$.
4. Numerically compute the central value for $r = 0.8$ and assert that $W_{\text{out}}(0, 0) < -0.1$.



In [ ]:
# Numerical Verification of Challenges 1, 2, and 3: CV Teleportation Benchmarks

engine = CVTeleportationEngine(squeezing_r=1.0)

# -------------------------------------------------------------
# Part 1: Classical Teleportation Limit F = 0.5 (Challenge 1)
# -------------------------------------------------------------
F_class = engine.coherent_state_fidelity(0.0)
print(f"Classical Teleportation Limit Check (r = 0):")
print(f"- Fidelity F(0): {F_class:.12f} (Expected: 0.5)")
assert np.isclose(F_class, 0.5, atol=1e-15), f"Classical limit mismatch: {F_class} != 0.5!"

# -------------------------------------------------------------
# Part 2: Quantum Fidelity & Noise Variance Benchmark (Challenge 2)
# -------------------------------------------------------------
r_target = 1.0
F_quantum = engine.coherent_state_fidelity(r_target)
expected_F = 1.0 / (1.0 + np.exp(-2.0))  # 0.8807970779778824

print(f"\nQuantum Advantage Fidelity Benchmark (r = {r_target}):")
print(f"- Numerical F(1.0):  {F_quantum:.12f}")
print(f"- Expected F:       {expected_F:.12f}")
assert np.isclose(F_quantum, expected_F, atol=1e-15), "Quantum fidelity mismatch!"
assert F_quantum > 0.5, "Failed to achieve quantum advantage over classical limit!"

# Stochastically verify noise variance with 50,000 shots
sim_shots = engine.teleport_quadratures(0.0, 0.0, n_shots=50000, seed=777)
emp_var = 0.5 * (sim_shots["var_noise_x"] + sim_shots["var_noise_p"])
expected_var = np.exp(-2.0 * r_target)
rel_err_var = np.abs(emp_var - expected_var) / expected_var

print(f"- Empirical Noise Variance: {emp_var:.6f}")
print(f"- Analytical e^(-2r):      {expected_var:.6f}")
print(f"- Relative Error:          {rel_err_var:.2e}")
assert rel_err_var < 0.02, f"Stochastic noise variance deviates by > 2%: {rel_err_var:.2e}!"

# -------------------------------------------------------------
# Part 3: Wigner Negativity Survival Benchmark (Challenge 3)
# -------------------------------------------------------------
r_neg = 0.8
var_neg = np.exp(-2.0 * r_neg)
# Analytical central Wigner value: (1/pi) * (var - 1) / (var + 1)^2
W0_analytic = (1.0 / np.pi) * (var_neg - 1.0) / (var_neg + 1.0)**2

print(f"\nWigner Negativity Survival Check (r = {r_neg}):")
print(f"- Analytical W_out(0, 0): {W0_analytic:.6f} (Negative if < 0)")

assert W0_analytic < -0.15, f"Wigner central value not sufficiently negative: {W0_analytic:.4f} >= -0.15!"

print("=" * 70)
print("SUCCESS: CV Quantum Teleportation Numerically Verified & Benchmarked!")
print(f"- Classical limit:      F_class = {F_class:.6f} == 0.500000 (PASS)")
print(f"- Quantum fidelity:     F(1.0) = {F_quantum*100:.2f}% (PASS: > 50%)")
print(f"- Channel noise check:  Var = {emp_var:.6f} vs. {expected_var:.6f} (PASS: < 2%)")
print(f"- Wigner negativity:    W(0,0) = {W0_analytic:.4f} < 0 (PASS)")
print("=" * 70)



---

## Summary & Next Steps

### Key Takeaways from Notebook 5.4
1. **The Braunstein-Kimble Protocol:** Harnesses continuous EPR correlations to teleport continuous quantum fields via joint homodyne Bell measurement and feedforward displacement.
2. **Exponential Noise Suppression:** Excess noise added by teleportation scales strictly as $\sigma_{\text{noise}}^2 = e^{-2r}$, vanishing asymptotically in the infinite-squeezing limit.
3. **The Classical Bound $\mathcal{F} \le 1/2$:** Rooted in the quantum no-cloning theorem, classical measure-and-prepare methods cannot exceed $50\%$ fidelity for coherent states.
4. **Quantum Advantage:** Any shared squeezing $r > 0$ breaks through the classical bound ($\mathcal{F} > 0.5$), enabling high-fidelity quantum communication.

---

### Up Next: Notebook 5.5
In [Notebook 5.5: Gaussian States, Symplectic Operations & Covariance Matrices](05_gaussian_states_and_symplectic_operations.ipynb), we introduce the formal mathematical architecture of **Gaussian quantum information**, studying the real symplectic group $\text{Sp}(2N, \mathbb{R})$, Williamson's theorem, covariance matrices, and Gaussian entanglement criteria (logarithmic negativity).

